In [1]:
import numpy as np
import os
import sys

# Navigate to the parent directory of the project structure
project_dir = os.path.abspath(os.path.join(os.getcwd(), '../..'))
src_dir = os.path.join(project_dir, 'src')
log_dir = os.path.join(project_dir, 'log')
fig_dir = os.path.join(project_dir, 'fig')
data_dir = os.path.join(project_dir, 'build')
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

# Add the src directory to sys.path
sys.path.append(src_dir)


from boolean_circuit.graph_synthesizer import CircuitGraph
from boolean_circuit.partitioner import KaHIPPartitioner

In [2]:
number_of_clients = 1024
boolean_circuit_file = os.path.join(data_dir, f'boolean_circuits/oblivious_sort_u32_N{number_of_clients}/output.gate.txt')
assert os.path.exists(boolean_circuit_file), f"File {boolean_circuit_file} does not exist"

cg = CircuitGraph.from_cbmc_gc_gate_file(boolean_circuit_file)
pg = cg.to_partitionable()

In [3]:
# partitioner = BalancedContiguousPartitioner()
seed = np.random.randint(0, 2**31)
partitioner = KaHIPPartitioner(mode=0, seed=2, suppress_output=0)  
part = partitioner.partition(pg, nparts=1024, gamma=0.1)
metrics = pg.summary_metrics(part, nparts=1024)

print("cut_pin:", metrics["cut_pin"])
print("max_cross_boundary:", metrics["max_cross_boundary"])
print("max_out_boundary:", metrics["max_out_boundary"])
print("max_in_boundary:", metrics["max_in_boundary"])
print("load:", metrics["load"])


cut_pin: 1172032
max_cross_boundary: 3632
max_out_boundary: 2379
max_in_boundary: 1953
load: [1911. 1935. 1934. ... 1796. 1935. 1829.]


In [4]:
print("client_balance_computation_cost:", metrics["max_client_balance_computation_cost (s)"])
print("client_balance_communication_cost:", metrics["max_client_balance_communication_cost (MB)"])
print("baseline computation cost:", metrics["baseline computation cost (s)"])
print("baseline communication cost:", metrics["baseline communication cost (MB)"])


client_balance_computation_cost: 36.640005754
client_balance_communication_cost: 1.442016
baseline computation cost: 327.68540672
baseline communication cost: 55.738368
